# LLM-as-judge

**Session 3 · Track A · local Ollama**

Grade open-ended outputs with a model, using a rubric and structured output.

In [2]:
import sys; sys.path.append('..')  # so `utils` and `eval` import from the repo root
import json
from utils import ask


In [3]:
def judge(answer, context):
    prompt = f"""Score the ANSWER 1-5 for faithfulness to the CONTEXT.
Return ONLY JSON: {{"score": int, "reason": str}}.

CONTEXT:
{context}

ANSWER:
{answer}"""
    raw = ask(prompt)
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"score": None, "reason": "unparseable: " + raw[:80]}

print(judge("Paris is the capital.", "France is a country in Europe. Its capital is Paris."))

{'score': 5, 'reason': 'The answer correctly states that Paris is the capital, which directly aligns with the information provided in the context.'}


### Worked example

Validate the judge against answers you label yourself, then wrap it into an eval scorer with the `(output, expected) -> bool` signature.


In [4]:
# Worked example: validate the judge, then use it as a scorer
LABELLED = [
    # (answer, context, should_pass)
    ("Paris is the capital of France.",          "France's capital is Paris.",        True),
    ("The capital is Lyon.",                     "France's capital is Paris.",        False),
    ("The city was founded in 1200.",            "The city was founded in 1150.",     False),
    ("Water boils at 100C at sea level.",        "At sea level water boils at 100C.", True),
    ("I am not sure.",                           "The report is due Friday.",         False),
]

agree = 0
for answer, ctx, gold_ok in LABELLED:
    v = judge(answer, ctx)
    ok = (v.get("score") or 0) >= 4
    agree += ok == gold_ok
    print(f"  score={v.get('score')} pass={ok} (want {gold_ok}) | {answer[:40]}")
print(f"\njudge agrees with me on {agree}/{len(LABELLED)}")

def judge_scorer(output, expected):
    """Adapt judge() into an eval scorer: expected is the reference context."""
    return (judge(output, expected).get("score") or 0) >= 4


  score=5 pass=True (want True) | Paris is the capital of France.
  score=1 pass=False (want False) | The capital is Lyon.
  score=1 pass=False (want False) | The city was founded in 1200.
  score=5 pass=True (want True) | Water boils at 100C at sea level.
  score=1 pass=False (want False) | I am not sure.

judge agrees with me on 5/5


## Your turn - vary the example

1. Add 3 borderline answers where you and the judge might disagree; check agreement.
2. Tighten the rubric in `judge()` (define what a 3 vs a 4 means) and re-check.
3. Run `run_eval(cases, my_answer_fn, scorer=judge_scorer)` on a small set.


In [ ]:
# Your variation here - copy the worked example above and change ONE thing, then re-run
